# 🔍 ERSeP — Análisis Manual en Colab
Usá este notebook para:
- Revisar el Excel generado por GitHub Actions
- Re-ejecutar el scraper manualmente sobre fechas específicas
- Analizar y filtrar los datos extraídos

In [ ]:
# ── 1. Instalar dependencias ──────────────────────────────────────────────
!pip install requests beautifulsoup4 pdfplumber openpyxl -q

In [ ]:
# ── 2. Subir el scraper.py desde tu PC o clonarlo desde GitHub ───────────
# OPCIÓN A: Subir manualmente (ejecutá esta celda y usá el botón que aparece)
from google.colab import files
uploaded = files.upload()   # subí scraper.py desde tu PC

# OPCIÓN B (recomendada): clonar tu repo de GitHub
# !git clone https://github.com/TU_USUARIO/TU_REPO.git
# %cd TU_REPO

In [ ]:
# ── 3. Ejecutar el scraper (últimos 7 días como prueba rápida) ────────────
# Cambiá el número para buscar más días
!python scraper.py 7

In [ ]:
# ── 4. Ver los resultados en Colab ────────────────────────────────────────
import pandas as pd
import json

# Leer el Excel generado
try:
    df = pd.read_excel('resultados_ersep.xlsx', sheet_name='Resoluciones ERSeP')
    print(f'✅ {len(df)} registros encontrados')
    display(df)
except FileNotFoundError:
    print('⚠️ Aún no se generó el Excel. Ejecutá la celda anterior primero.')

# Ver resumen JSON
with open('resumen.json') as f:
    resumen = json.load(f)
print('\n=== RESUMEN ===')
print(json.dumps(resumen, indent=2, ensure_ascii=False))

In [ ]:
# ── 5. Filtros y análisis adicionales ─────────────────────────────────────
if 'df' in dir() and len(df) > 0:
    # Filtrar por prestadora específica
    # df_filtrado = df[df['Prestadora'].str.contains('VILLA GENERAL', na=False)]

    # Ver resoluciones con porcentaje de aumento
    con_pct = df[df['% Otorgado'].notna()]
    print(f'Resoluciones con % otorgado detectado: {len(con_pct)}')
    display(con_pct[['Fecha Boletín', 'Resolución', 'Prestadora', '% Otorgado', 'Rige Desde']])
else:
    print('No hay datos para analizar todavía.')

In [ ]:
# ── 6. Descargar el Excel a tu PC ─────────────────────────────────────────
from google.colab import files
files.download('resultados_ersep.xlsx')

In [ ]:
# ── 7. (AVANZADO) Analizar un PDF específico manualmente ─────────────────
import pdfplumber

# Subí el PDF manualmente
from google.colab import files
uploaded_pdf = files.upload()   # subí el PDF que querés inspeccionar

pdf_nombre = list(uploaded_pdf.keys())[0]

with pdfplumber.open(pdf_nombre) as pdf:
    for i, pagina in enumerate(pdf.pages, 1):
        texto = pagina.extract_text() or ''
        if 'ERSeP' in texto or 'Resolución General' in texto:
            print(f'\n=== PÁGINA {i} ===')
            print(texto[:1500])
            print('---')